# Customer Churn & Retention Analytics — Step 2: Clean the Dataset

Based on what I found in Step 1, there are four things to fix here:
1. `TotalCharges` is stored as text and has 11 blank values (all at tenure = 0) — convert to numeric, fill blanks with 0
2. `SeniorCitizen` is 0/1 instead of Yes/No like every other binary column — standardize it
3. A handful of service columns use `'No internet service'` / `'No phone service'` as a third category — decide how to handle this
4. General column name / structure check before moving into EDA

Loading the raw file fresh so this notebook works standalone from Step 1.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.shape

(7043, 21)

## Fix 1: TotalCharges

In [2]:
# customers with tenure = 0 have no accumulated billing history yet,
# so a blank TotalCharges there means 0, not truly missing data.
# doing this conditionally (rather than a blanket fillna(0)) so this only
# touches the specific rows I've actually checked, not any other NaN that
# might show up here for a different reason
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df.loc[(df['tenure'] == 0) & (df['TotalCharges'].isna()), 'TotalCharges'] = 0

# double check the fix worked and the 11 rows are now 0 instead of NaN
print(df['TotalCharges'].dtype)
print("remaining NaNs after the targeted fill:", df['TotalCharges'].isna().sum())
print(df.loc[df['tenure'] == 0, ['customerID', 'tenure', 'TotalCharges']])

float64
remaining NaNs after the targeted fill: 0
      customerID  tenure  TotalCharges
488   4472-LVYGI       0           0.0
753   3115-CZMZD       0           0.0
936   5709-LVOEQ       0           0.0
1082  4367-NUYAO       0           0.0
1340  1371-DWPAZ       0           0.0
3331  7644-OMVMY       0           0.0
3826  3213-VVOLG       0           0.0
4380  2520-SGTTA       0           0.0
5218  2923-ARZLG       0           0.0
6670  4075-WKNIU       0           0.0
6754  2775-SEFEE       0           0.0


## Fix 2: SeniorCitizen 0/1 → Yes/No\n\nEvery other binary column in this dataset (`Partner`, `Dependents`, `PhoneService`, etc.) uses 'Yes'/'No' strings. `SeniorCitizen` is the odd one out with 0/1, which would be inconsistent if I'm grouping by multiple columns together later.

In [3]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['SeniorCitizen'].value_counts()

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64

## Fix 3: 'No internet service' / 'No phone service' categories

Six columns have this pattern: `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`. Decision here: **keep these as their own category rather than collapsing to just 'No'**, because they're not actually the same thing — a customer with internet who chose not to buy security add-ons is a different case than a customer who doesn't have internet at all. Collapsing them would hide a real distinction that matters for the "why are they leaving" analysis (e.g. add-on gaps only make sense to look at among customers who *have* internet in the first place).

Just confirming the columns and their category counts here so this decision is documented, not changing anything.

In [4]:
service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- MultipleLines ---
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

--- OnlineSecurity ---
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

--- OnlineBackup ---
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

--- DeviceProtection ---
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

--- TechSupport ---
TechSupport
No                     3473
Yes                    2044
No internet service    1526
Name: count, dtype: int64

--- StreamingTV ---
StreamingTV
No                     2810
Yes                    2707
No internet service    1526
Name: count, dtype: int64

--- StreamingMovies ---
StreamingMovies
No                     2785
Yes                    2732
No internet service    1526
Name:

## Fix 4: column names and final structure check

In [5]:
# column names are already clean (no spaces, consistent CamelCase) so nothing to rename
print(df.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   object 
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


`TotalCharges` is now float64 and `SeniorCitizen` is now an object column with Yes/No, matching the rest of the binary columns. No more nulls, no more type mismatches.

## One more check: does the math add up?\n\nSanity check — `TotalCharges` should roughly track `tenure * MonthlyCharges` (not exact, since prices can change over time, but it shouldn't be wildly off).

In [7]:
df['expected_total'] = df['tenure'] * df['MonthlyCharges']
df['charge_diff_pct'] = ((df['TotalCharges'] - df['expected_total']).abs() / df['expected_total'].replace(0, np.nan)) * 100

print(df['charge_diff_pct'].describe())

# drop the two helper columns, they were just for this sanity check
df = df.drop(columns=['expected_total', 'charge_diff_pct'])

count    7032.000000
mean        3.201747
std         3.989999
min         0.000000
25%         0.721968
50%         1.999740
75%         4.173428
max        57.345361
Name: charge_diff_pct, dtype: float64


Median difference is small and the distribution looks reasonable — no red flags suggesting a bigger data issue. Makes sense that it's not a perfect match since monthly rates likely changed for some customers over their tenure.

## Outlier check

Should have done this earlier instead of just eyeballing `.describe()` — running an actual IQR check on the three numeric columns before calling the cleaning done.

In [8]:
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: range [{df[col].min()}, {df[col].max()}], IQR bounds [{lower:.2f}, {upper:.2f}], outliers: {len(outliers)}")

tenure: range [0, 72], IQR bounds [-60.00, 124.00], outliers: 0
MonthlyCharges: range [18.25, 118.75], IQR bounds [-46.02, 171.38], outliers: 0
TotalCharges: range [0.0, 8684.8], IQR bounds [-4683.52, 8868.67], outliers: 0


No outliers flagged on any of the three columns. Makes sense given the nature of the data — `tenure` is capped at 72 months (6 years, likely the max in whatever time window this data covers), and both charge columns are bounded by real plan pricing rather than being open-ended, so there's no room for data-entry errors or extreme junk values to show up.

## Final validation before export

One last check across the board before this becomes the file every later step depends on.

In [9]:
print("Missing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDataset shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

Missing values:
0

Duplicate rows:
0

Dataset shape:
(7043, 21)

Data types:
customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


## Save the cleaned dataset\n\nExporting this so Step 3 (EDA) and later steps (SQL load, ML) all read from the same cleaned file instead of re-doing this cleaning each time.

In [10]:
df.to_csv('telco_churn_cleaned.csv', index=False)
print(f"Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Saved cleaned dataset: 7043 rows, 21 columns


## Summary

- `TotalCharges` converted to numeric, 11 blank values (all tenure = 0, new customers) filled with 0
- `SeniorCitizen` standardized from 0/1 to Yes/No to match the rest of the binary columns
- Kept the `'No internet service'` / `'No phone service'` categories as-is rather than collapsing them — they carry real information for the churn analysis
- Sanity-checked `TotalCharges` against `tenure × MonthlyCharges` — no major inconsistencies
- Saved as `telco_churn_cleaned.csv` for use in every step from here on

Next step: EDA — actually digging into the churn patterns visually and statistically.